In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

In [21]:
feature_vectors = pd.read_csv('../../../Microglial_Morphology_freeze_run/03_morph_embedding/feature_vectors_texture.csv')
microglia = sc.read_h5ad('Transciptomic_labels_and_morphology_labels_full.h5ad')

In [22]:
feature_vectors.columns

Index(['Cell Area', 'Cell Perimeter', 'Convex Hull Area',
       'Convex Hull Perimeter', 'Cell Solidity', 'Cell Convexity',
       'Cell Roughness', 'Cell Circularity', 'Convex Hull Span Ratio',
       'Convex Hull Circularity', 'Soma Area', 'Soma Perimeter',
       'Soma Circularity', 'Soma Ratio', 'Skeleton Length',
       'Mean Branch Length', 'Number of Branches',
       'Number of Branching Points', 'Number of Terminal Points',
       'Branching Index', 'Dendritic Maximum', 'Ramification Index',
       'Radius of Influence', 'Fractal Dimension', 'Euclidean Distance',
       'Path Distance', 'Tortuosity', 'Lacunarity', 'Eccentricity',
       'Normalized Intensity', 'DAPI Score', 'embed_raw', 'embed_segment',
       'Name', 'Critical Radius'],
      dtype='object')

In [23]:
# trim to the cells used for analysis
feature_vectors = feature_vectors[feature_vectors.Name.isin(microglia.obs.Name.tolist())]

# trim away non-morphology columns
columns_to_drop = ['embed_raw', 'embed_segment', 'Critical Radius', 'DAPI Score', 'Normalized Intensity']
feature_vectors = feature_vectors.drop(columns=[col for col in columns_to_drop if col in feature_vectors.columns])

print(feature_vectors.shape)
designed_trim = set(feature_vectors.columns.tolist())

# making sure we still have the same object if we get rid of NaNs
feature_vectors = feature_vectors.dropna(axis=1, how='any')
nan_drop = set(feature_vectors.columns.tolist())

print(feature_vectors.shape)
print(designed_trim - nan_drop)

(3949, 30)
(3949, 29)
{'Ramification Index'}


In [30]:
microglia.obs.ordered_morph.value_counts()

ordered_morph
3    1160
2    1079
4     712
1     539
0     459
Name: count, dtype: int64

In [38]:
C1 = feature_vectors[feature_vectors.Name.isin(microglia[microglia.obs.ordered_morph == '0'].obs.Name.tolist())].set_index('Name')
C2 = feature_vectors[feature_vectors.Name.isin(microglia[microglia.obs.ordered_morph == '1'].obs.Name.tolist())].set_index('Name')
C3 = feature_vectors[feature_vectors.Name.isin(microglia[microglia.obs.ordered_morph == '2'].obs.Name.tolist())].set_index('Name')
C4 = feature_vectors[feature_vectors.Name.isin(microglia[microglia.obs.ordered_morph == '3'].obs.Name.tolist())].set_index('Name')
C5 = feature_vectors[feature_vectors.Name.isin(microglia[microglia.obs.ordered_morph == '4'].obs.Name.tolist())].set_index('Name')

In [41]:
morphologies = [C1,C2,C3,C4,C5]
with pd.ExcelWriter('Export_CSVs/morphology_vectors.xlsx', engine='openpyxl') as writer:
    morphologies[0].to_excel(writer, sheet_name='C1', index=True)
    morphologies[1].to_excel(writer, sheet_name='C2', index=True)
    morphologies[2].to_excel(writer, sheet_name='C3', index=True)
    morphologies[3].to_excel(writer, sheet_name='C4', index=True)
    morphologies[4].to_excel(writer, sheet_name='C5', index=True)